<a href="https://colab.research.google.com/github/team0243/Project_ML/blob/main/Statistical_analysis_RCC_UCUT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import shapiro, ttest_ind, mannwhitneyu
from scipy.stats import pearsonr, spearmanr, shapiro

In [ ]:
url = 'https://github.com/team0243/Project_ML/blob/main/Dataset_RCC_UTUC_ML.xlsx?raw=true'
try:
  df_all = pd.read_excel(url)
  # Convert all columns to numeric, errors='coerce' will replace non-numeric values with NaN
  df = df_all.apply(pd.to_numeric, errors='coerce')
  # Drop columns with all NaN values after conversion
  df = df_all.dropna(axis=1, how='all')
except Exception as e:
  print(f"An error occurred: {e}")
  print("Please ensure the link is correct, the file exists, and the proper permissions are set.")

In [ ]:
df_all.head()

In [ ]:
df_all.describe()

วิเคราะห์ความสัมพันธ์ระหว่างตัวแปร

In [ ]:
y_variables = ['NLR', 'PLR', 'WBC', 'PLT', 'NE%', 'LY%', 'Lymphocytes', 'Neutrophil']

for y_var in y_variables:
  if y_var in df.columns:
    plt.figure(figsize=(8, 6))
    # Print the available columns to check for the correct name of the 'Age' column
    print(df.columns)
    # Replace 'Age' with the actual column name from the printed list if it's different
    sns.scatterplot(x=df.columns[0], y=y_var, data=df)
    plt.title(f'Scatter Plot of Age vs. {y_var}')
    plt.xlabel(df.columns[0])  # Replace with actual column name
    plt.ylabel(y_var)
    plt.show()
  else:
    print(f"Warning: Column '{y_var}' not found in the DataFrame.")

In [ ]:
#ทดสอบการแจกแจงปกติของตัวแปร
variables = ['NLR', 'PLR', 'WBC', 'PLT', 'NE%', 'LY%', 'Lymphocytes', 'Neutrophil']
normality_results = {}
for var in variables:
    stat, p_value = shapiro(df[var])
    normality_results[var] = p_value

# แสดงผลการทดสอบ Normality
print("\n ผลการทดสอบ Normality (Shapiro-Wilk Test)")
for var, p in normality_results.items():
    print(f"{var}: p-value = {p:.4f} {'✅ Normal' if p > 0.05 else '❌ Not Normal'}")


In [ ]:
x = df[df.columns[0]]  # Assuming 'Age' is the first column. Change the index if needed

In [ ]:
#คำนวณ Pearson หรือ Spearman Correlation ตามเงื่อนไข
correlation_results = []
for var in variables:
    if normality_results[var] > 0.05:  # ถ้าข้อมูลเป็นปกติ ใช้ Pearson
        corr, p_value = pearsonr(df[var], x)
        method = "Pearson"
    else:  # ถ้าข้อมูลไม่เป็นปกติ ใช้ Spearman
        corr, p_value = spearmanr(df[var], x)
        method = "Spearman"

    correlation_results.append([var, method, corr, p_value])

# สร้าง DataFrame แสดงผลลัพธ์
df_correlation = pd.DataFrame(correlation_results, columns=["Variable", "Method", "Correlation", "p-value"])
print("\n📊 ผลการวิเคราะห์ความสัมพันธ์ระหว่างตัวแปร Age กับตัวแปรอื่นๆ:")
print(df_correlation)

In [ ]:
# Select only numeric columns for correlation calculation
#create Pearson Correlation Matrix
numeric_df = df.select_dtypes(include=['number'])
corr_matrix = numeric_df.corr()

# Set the graph size
plt.figure(figsize=(10, 8))

# create Heatmap
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)

# Visualize data
plt.title("Correlation Matrix")
plt.show()

เปรียบเทียบค่าดัชนีทางโลหิตวิทยาระหว่างกลุ่ม RCC กับ UTUC

In [ ]:
url = 'https://github.com/team0243/Project_ML/blob/main/Dataset_RCC_UTUC_ML.xlsx?raw=true'
try:
  df2 = pd.read_excel(url)
except Exception as e:
  print(f"An error occurred: {e}")
  print("Please ensure the link is correct, the file exists, and the proper permissions are set.")

In [ ]:
# แยกกลุ่มข้อมูลตาม Diagnosis
df_RCC = df2[df2['Diagnosis'] == 'RCC']
df_UTUC = df2[df2['Diagnosis'] == 'UTUC']

# กำหนดชื่อของดัชนีทางโลหิตวิทยาที่ต้องการเปรียบเทียบ
hematology_indices = ['NLR', 'PLR', 'WBC', 'PLT', 'NE%', 'LY%', 'Lymphocytes', 'Neutrophil']

In [ ]:
# วนลูปทดสอบสำหรับแต่ละตัวแปร
for var in hematology_indices:
    print(f"\n--- วิเคราะห์ตัวแปร: {var} ---")

    # ตรวจสอบ normality ด้วย Shapiro-Wilk Test สำหรับแต่ละกลุ่ม
    stat_RCC, p_RCC = shapiro(df_RCC[var])
    stat_UTUC, p_UTUC = shapiro(df_UTUC[var])

    print(f"Shapiro-Wilk test: RCC p = {p_RCC:.3f}, UTUC p = {p_UTUC:.3f}")

    # ถ้าข้อมูลในทั้งสองกลุ่มแจกแจงปกติ (p > 0.05) ใช้ T-test
    if p_RCC > 0.05 and p_UTUC > 0.05:
        stat, p_value = ttest_ind(df_RCC[var], df_UTUC[var])
        test_used = "T-test"
    else:
        stat, p_value = mannwhitneyu(df_RCC[var], df_UTUC[var])
        test_used = "Mann-Whitney U test"

    print(f"{test_used} สำหรับ {var}: p-value = {p_value:.3f}")

    if p_value < 0.05:
        print(f"→ มีความแตกต่างอย่างมีนัยสำคัญใน {var} ระหว่าง RCC กับ UTUC")
    else:
        print(f"→ ไม่มีความแตกต่างอย่างมีนัยสำคัญใน {var} ระหว่าง RCC กับ UTUC")


ผลลัพธ์ที่ได้จากการวิเคราะห์เปรียบเทียบค่าดัชนีโลหิตวิทยาระหว่างกลุ่ม RCC และ UTUC

In [ ]:
# Data for the results
data = {
    "Variable": ["NLR", "PLR", "WBC", "PLT", "NE%", "LY%", "Lymphocytes", "Neutrophil"],
    "Shapiro-Wilk p-value (RCC)": [0.000, 0.000, 0.000, 0.000, 0.046, 0.001, 0.004, 0.000],
    "Shapiro-Wilk p-value (UTUC)": [0.000, 0.000, 0.000, 0.003, 0.212, 0.047, 0.000, 0.000],
    "Mann-Whitney U p-value": [0.563, 0.765, 0.530, 0.269, 0.677, 0.547, 0.139, 0.902],
    "Result": [
        "No significant difference in NLR between RCC and UTUC",
        "No significant difference in PLR between RCC and UTUC",
        "No significant difference in WBC between RCC and UTUC",
        "No significant difference in PLT between RCC and UTUC",
        "No significant difference in NE% between RCC and UTUC",
        "No significant difference in LY% between RCC and UTUC",
        "No significant difference in Lymphocytes between RCC and UTUC",
        "No significant difference in Neutrophil between RCC and UTUC"
    ]
}

# Create DataFrame
df_results = pd.DataFrame(data)

# Display the table
df_results


In [ ]:
#save df_results  as csv file

df_results.to_csv('df_results.csv', index=False)
